In [1]:

import os
import time
import json
import pandas as pd

from pydantic import BaseModel, Field, field_validator, ValidationError


def repo_root():
    cur = os.getcwd()
    for _ in range(8):
        if os.path.isdir(os.path.join(cur, 'data')) and os.path.exists(
            os.path.join(cur, 'data', 'appointments.csv')
        ):
            return cur
        nxt = os.path.dirname(cur)
        if nxt == cur:
            break
        cur = nxt
    raise FileNotFoundError('Could not locate the repository root')


ROOT = repo_root()
OUTPUT_DIR = ROOT
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------------
# Python 3.9 shim: google-genai transitively imports the `mcp` package,
# which requires Python 3.10+. The runtime only needs `mcp` under a
# try/except, so we inject a stub that raises ImportError for `mcp.types`,
# letting google-genai degrade gracefully (mcp is not used at runtime).
# ------------------------------------------------------------------
import sys as _sys
import types as _types

_mcp_stub = _types.ModuleType('mcp')


def _mcp_getattr(name):
    if name == 'types' or name == 'ClientSession':
        raise ImportError('no mcp.' + name)
    raise AttributeError(name)


_mcp_stub.__getattr__ = _mcp_getattr
_sys.modules.setdefault('mcp', _mcp_stub)

print('Repository root:', ROOT)

# Allowed root causes (fixed vocabulary).
ALLOWED_ROOT_CAUSES = [
    'forgot_or_no_reminder',
    'distance_or_transport',
    'cost_or_fee',
    'illness_or_emergency',
    'work_or_schedule',
    'other',
]

# JSON schema for the Gemini structured response.
JSON_SCHEMA = {
    'type': 'object',
    'properties': {
        'root_cause': {
            'type': 'string',
            'description': 'One of the allowed root causes.',
        },
        'confidence': {
            'type': 'number',
            'description': 'Confidence score between 0 and 1.',
        },
        'summary': {
            'type': 'string',
            'description': 'Short plain-language summary of the note, at most 25 words.',
        },
        'recommended_action': {
            'type': 'string',
            'description': 'Recommended follow-up action for the call centre.',
        },
    },
    'required': ['root_cause', 'confidence', 'summary', 'recommended_action'],
}

API_CALL_COUNT = 0
api_key = None
genai_client = None


# Exact JSON text of the most recent response, keyed by note text.
# Kept separate so analyse_note() can return a clean dictionary while the
# processing loop still preserves every raw model response verbatim.
RAW_RESPONSES = {}


# Notes are loaded once here and reused by Tasks 4.1, 4.3 and 4.4.
notes_df = pd.read_csv(os.path.join(ROOT, 'data', 'followup_notes.csv'))


Repository root: /Users/harish/careloop-clinics-no-show-guard


## Task 4.1 — Structured Gemini call

Objective: classify a single call-centre note with Google Gemini Flash using structured JSON output.

- Model: Gemini Flash (`gemini-2.5-flash`).
- `temperature=0`.
- JSON response mode with a JSON schema.
- System instruction.
- CRAFT prompt.
- Minimal thinking configuration (thinking budget 0) where supported.
- `max_output_tokens >= 1000`.
- The API key is entered interactively at runtime via `input()`; it is never hardcoded and never written to a file.

The schema contains exactly `root_cause`, `confidence`, `summary`, `recommended_action`. `root_cause` remains a plain string.

Allowed root causes:
`forgot_or_no_reminder`, `distance_or_transport`, `cost_or_fee`, `illness_or_emergency`, `work_or_schedule`, `other`.

First, N001 is tested. Its expected root cause is `forgot_or_no_reminder`.

In [3]:
# In Google Colab, uncomment the next line before running this notebook:
# !pip install -q google-genai pydantic

import json
from google import genai
from google.genai import types


def craft_prompt(note_text):
    """CRAFT prompt: Context, Role, Action, Tone, Format."""
    allowed = ', '.join(f'`{c}`' for c in ALLOWED_ROOT_CAUSES)
    return (
        "Context: You are a care-coordination analyst at CareLoop Clinics. "
        "Call-centre notes record why a patient missed an outpatient appointment.\n"
        "Role: You classify the root cause of each missed appointment.\n"
        "Action: Read the note and choose exactly one allowed root cause: " + allowed + ". "
        "If the note is ambiguous, answer `other` rather than guessing.\n"
        "Tone: Neutral and factual; do not speculate beyond the note.\n"
        "Format: Return only one JSON object with root_cause, confidence (0 to 1), "
        "summary (at most 25 words) and recommended_action.\n\n"
        f"Note: {note_text}"
    )


SYSTEM_INSTRUCTION = (
    "You are a care-coordination analyst. Classify call-centre notes into one "
    "root cause and return a single JSON object. Do not write text outside the JSON."
)


def analyse_note(note_text, model='gemini-flash-lite-latest'):
    """Return the parsed response dictionary with exactly the four schema fields.

    The exact JSON text is also stored in RAW_RESPONSES[note_text] so the caller
    can preserve it in the raw_response column.
    """
    global API_CALL_COUNT
    API_CALL_COUNT += 1
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_INSTRUCTION,
        temperature=0,
        max_output_tokens=1000,
        response_mime_type='application/json',
        response_schema=JSON_SCHEMA,
    )
    try:  # minimal thinking level where the model supports it
        config.thinking_config = types.ThinkingConfig(thinking_level='minimal')
    except Exception:
        pass  # if the model rejects it, keep all other settings unchanged

    response = genai_client.models.generate_content(
        model=model,
        contents=craft_prompt(note_text),
        config=config,
    )
    raw = response.text
    RAW_RESPONSES[note_text] = raw
    data = json.loads(raw)
    return {k: data[k] for k in ['root_cause', 'confidence', 'summary', 'recommended_action']}


# Enter the key at run time (never stored in a file or notebook output).
api_key = input('Paste your key: ').strip()
if not api_key:
    raise ValueError('No API key entered.')
genai_client = genai.Client(api_key=api_key)
print('Gemini client initialised.')

n001_text = notes_df[notes_df['note_id'] == 'N001'].iloc[0]['note_text']
print('N001 note:', n001_text)
n001_result = analyse_note(n001_text)
print('N001 parsed:', n001_result)
print('API calls so far:', API_CALL_COUNT)


Gemini client initialised.
N001 note: Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.
N001 parsed: {'root_cause': 'forgot_or_no_reminder', 'confidence': 1.0, 'summary': 'Patient forgot the appointment because she did not receive any reminder message and requested to reschedule for next week.', 'recommended_action': "Reschedule the appointment for next week and verify the patient's contact details for future reminders."}
API calls so far: 1


## Task 4.2 — Pydantic validation

Objective: validate every Gemini response with Pydantic, process the first ten notes, and save `output/notes_analysed.csv`.

`FollowupInsight(BaseModel)`:
- `root_cause: str = Field(min_length=3)`
- `confidence: float = Field(ge=0, le=1)`
- `summary: str = Field(min_length=5)`
- `recommended_action: str = Field(min_length=5)`
- `root_cause` is validated against the allowed list.

Input validation:
- `pd.isna(note_text)` is checked **before** calling `.strip()`.
- Missing, empty, or whitespace-only notes are **not** sent to Gemini.
  They are recorded as `root_cause = other`, `confidence = 0.0`, `summary = 'No note recorded'`, `recommended_action = 'Re-call patient'`, `validation_status = 'skipped_blank'`.
- N004 (empty note) is therefore skipped without an API request.
- For all other notes: call Gemini, validate with Pydantic, check the allowed root cause, record valid/invalid, and catch errors per note so one failure never stops the loop.
- Wait 13 seconds between API calls.

Expected API calls for Task 4.2: nine.

`output/notes_analysed.csv` columns: `note_id, appointment_id, root_cause, confidence, summary, recommended_action, validation_status, raw_response`. Every raw model response is preserved exactly as returned.

In [5]:
import time

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

    @field_validator('root_cause')
    @classmethod
    def _check_root_cause(cls, v):
        if v not in ALLOWED_ROOT_CAUSES:
            raise ValueError(f'root_cause must be one of {ALLOWED_ROOT_CAUSES}, got {v!r}')
        return v


COLUMNS_OUT = ['note_id', 'appointment_id', 'root_cause', 'confidence', 'summary',
               'recommended_action', 'validation_status', 'raw_response']

notes_ten = pd.read_csv(os.path.join(ROOT, 'data', 'followup_notes.csv')).head(10)
results = []
call_started = False

for _, note in notes_ten.iterrows():
    note_id = note['note_id']
    appointment_id = note['appointment_id']
    note_text = note['note_text']

    # Input validation: check pd.isna before any string operation.
    if pd.isna(note_text) or str(note_text).strip() == '':
        results.append({
            'note_id': note_id, 'appointment_id': appointment_id,
            'root_cause': 'other', 'confidence': 0.0,
            'summary': 'No note recorded', 'recommended_action': 'Re-call patient',
            'validation_status': 'skipped_blank', 'raw_response': '',
        })
        print(f'{note_id}: blank note — skipped without an API call')
        continue

    if call_started:
        time.sleep(13)
    call_started = True

    raw = ''
    data = {}
    try:
        data = analyse_note(note_text)
        raw = RAW_RESPONSES.get(note_text, '')
        insight = FollowupInsight(**data)
        status = 'valid'
        root_cause = insight.root_cause
        confidence = insight.confidence
        summary = insight.summary
        action = insight.recommended_action
        print(f'{note_id}: {root_cause} ({confidence}) — {status}')
    except Exception as exc:
        # Keep the reply as it came back; copy only fields it actually contained.
        status = 'invalid' if data else f'error: {type(exc).__name__}'
        root_cause = data.get('root_cause', '')
        confidence = data.get('confidence', '')
        summary = data.get('summary', '')
        action = data.get('recommended_action', '')
        print(f'{note_id}: {status} — {exc}')

    results.append({
        'note_id': note_id, 'appointment_id': appointment_id,
        'root_cause': root_cause, 'confidence': confidence,
        'summary': summary, 'recommended_action': action,
        'validation_status': status, 'raw_response': raw,
    })

analysed = pd.DataFrame(results, columns=COLUMNS_OUT)
NOTES_CSV = os.path.join(ROOT, 'notes_analysed.csv')
analysed.to_csv(NOTES_CSV, index=False)
print()
print('Wrote', NOTES_CSV, '| rows:', len(analysed), '| columns:', len(analysed.columns))
print('Notes sent to Gemini:', int((analysed['validation_status'] != 'skipped_blank').sum()))
print('Total API calls so far:', API_CALL_COUNT)
print(analysed[['note_id', 'root_cause', 'validation_status']].to_string(index=False))


N001: forgot_or_no_reminder (1.0) — valid
N002: distance_or_transport (1.0) — valid
N003: cost_or_fee (0.95) — valid
N004: blank note — skipped without an API call
N005: illness_or_emergency (0.99) — valid
N006: work_or_schedule (1.0) — valid
N007: forgot_or_no_reminder (0.95) — valid
N008: error: ServerError — 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
N009: other (0.9) — valid
N010: work_or_schedule (0.95) — valid

Wrote /Users/harish/careloop-clinics-no-show-guard/notes_analysed.csv | rows: 10 | columns: 8
Notes sent to Gemini: 9
Total API calls so far: 10
note_id            root_cause  validation_status
   N001 forgot_or_no_reminder              valid
   N002 distance_or_transport              valid
   N003           cost_or_fee              valid
   N004                 other      skipped_blank
   N005  illness_or_emergency        

## Task 4.3 — Validation and prompt testing

Objective: exercise Pydantic validation on deliberately invalid inputs and compare two prompts on the same notes.

Pydantic tests:
1. `confidence = 1.7` (outside `0–1`).
2. Missing `recommended_action`.
Both raise `ValidationError`; the actual errors are printed.

Prompt comparison on N007 and N009:
- V1: `'Classify this note.'`
- V2: full CRAFT prompt with allowed root causes and an explicit instruction to return `other` rather than guess.
- Same model settings and schema for both.

The four results are shown in a Markdown table. API requests are counted across Tasks 4.1–4.3 (expected total approximately 12–14, below 25).

In [7]:
# (a) Deliberately invalid Pydantic inputs.
print('=== Pydantic validation tests ===')
try:
    FollowupInsight(root_cause='other', confidence=1.7,
                    summary='A note.', recommended_action='Call patient.')
    print('confidence=1.7: UNEXPECTEDLY ACCEPTED')
except ValidationError as e:
    print('confidence=1.7 -> ValidationError:')
    print(e)

try:
    FollowupInsight(root_cause='other', confidence=0.5, summary='A note.')
    print('missing recommended_action: UNEXPECTEDLY ACCEPTED')
except ValidationError as e:
    print('missing recommended_action -> ValidationError:')
    print(e)


def classify_with_prompt(note_text, prompt):
    """Call Gemini with a custom prompt, validate, return the result dict."""
    global API_CALL_COUNT
    API_CALL_COUNT += 1
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_INSTRUCTION,
        temperature=0,
        max_output_tokens=1000,
        response_mime_type='application/json',
        response_schema=JSON_SCHEMA,
    )
    try:
        config.thinking_config = types.ThinkingConfig(thinking_level='minimal')
    except Exception:
        pass
    response = genai_client.models.generate_content(
        model='gemini-flash-lite-latest',
        contents=prompt + '\n\nNote: ' + note_text,
        config=config,
    )
    data = json.loads(response.text)
    allowed = data.get('root_cause', '') in ALLOWED_ROOT_CAUSES
    return {
        'note_id': '', 'prompt': '', 'root_cause': data.get('root_cause', ''),
        'confidence': data.get('confidence', ''), 'ok': allowed,
    }


V1_PROMPT = 'Classify this note'
V2_PROMPT = craft_prompt('[note inserted below]')  # full CRAFT prompt, same as Task 4.1

comparison_rows = []
for note_id in ['N007', 'N009']:
    text = notes_df[notes_df['note_id'] == note_id].iloc[0]['note_text']
    for label, prompt in [('V1: Classify this note', V1_PROMPT),
                          ('V2: CRAFT prompt', V2_PROMPT)]:
        res = classify_with_prompt(text, prompt)
        res['note_id'] = note_id
        res['prompt'] = label
        comparison_rows.append(res)

print()
print('=== Prompt comparison (N007, N009) ===')
header = '| note_id | prompt | root_cause | confidence | valid |'
print(header)
print('|' + '-' * (len(header) - 2) + '|')
for r in comparison_rows:
    print('| {note_id} | {prompt} | {root_cause} | {confidence} | {ok} |'.format(**r))

print()
print('V2 is the prompt to deploy: it states the allowed values and instructs the '
      'model to answer `other` rather than guess, so its root_cause is checked '
      'against the allowed list.')
print()
print('Total API calls across Tasks 4.1-4.3:', API_CALL_COUNT)


=== Pydantic validation tests ===
confidence=1.7 -> ValidationError:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.12/v/less_than_equal
missing recommended_action -> ValidationError:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'other', '...5, 'summary': 'A note.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

=== Prompt comparison (N007, N009) ===
| note_id | prompt | root_cause | confidence | valid |
|----------------------------------------------------|
| N007 | V1: Classify this note | Technical Issue | 0.95 | False |
| N007 | V2: CRAFT prompt | forgot_or_no_reminder | 0.95 | True |
| N009 | V1: Classify this note | Unable to Reach Patient | 1.0 | False |
| N009 | V2: CRAFT prompt | other | 0.8 | 

## Task 4.4 — Final recommendation

Objective: merge the analysed notes with the cleaned appointments and examine the relationship between extracted root causes and whether a reminder was sent.

- Merge `notes_analysed.csv` with `appointments_clean.csv` on `appointment_id`.
- Verify the merged table has ten rows.
- `pd.crosstab(root_cause, reminder_sent)`.
- Discuss the relationship; do not overgeneralise from ten notes.

In [9]:
import pandas as pd

appointments_clean = pd.read_csv(os.path.join(ROOT, 'appointments_clean.csv'))
analysed = pd.read_csv(NOTES_CSV)

merged = analysed.merge(
    appointments_clean[['appointment_id', 'reminder_sent', 'status', 'clinic_id']],
    on='appointment_id', how='left',
)
print('Merged rows:', len(merged))
print('Expected: 10')
print()
print(merged[['note_id', 'appointment_id', 'root_cause', 'reminder_sent', 'status']].to_string(index=False))
print()
print('=== Crosstab: root_cause x reminder_sent ===')
print(pd.crosstab(merged['root_cause'], merged['reminder_sent']))
print()
n_no_reminder = int((merged['reminder_sent'] == 'No').sum())
print('Notes with reminder_sent = No:', n_no_reminder, 'of 10')
print()
print('Discussion: with only ten notes this is illustrative, not clinic-wide evidence. '
      'If forgetfulness clusters among appointments with no reminder, that supports '
      'prioritising reminder delivery and confirmation for at-risk bookings — '
      'consistent with the ML model association found in Part 3.')


Merged rows: 10
Expected: 10

note_id appointment_id            root_cause reminder_sent  status
   N001          A0872 forgot_or_no_reminder            No No-Show
   N002          A0897 distance_or_transport            No No-Show
   N003          A0526           cost_or_fee           Yes No-Show
   N004          A0926                 other            No No-Show
   N005          A0534  illness_or_emergency           Yes No-Show
   N006          A0507      work_or_schedule           Yes No-Show
   N007          A0333 forgot_or_no_reminder            No No-Show
   N008          A0222                   NaN           Yes No-Show
   N009          A0977                 other           Yes No-Show
   N010          A0596      work_or_schedule            No No-Show

=== Crosstab: root_cause x reminder_sent ===
reminder_sent          No  Yes
root_cause                    
cost_or_fee             0    1
distance_or_transport   1    0
forgot_or_no_reminder   2    0
illness_or_emergency    0    1
o